# Gromacs notebook

This notebook keeps only the local Gromacs step4 minimization path.

Even with the reduced compute requirement here (minimizations only),
GPUs are still required for practical batch throughput across a full set.

It assumes all required inputs already live in:
`/media/volume/sirna-features/gromacs`

Local inputs used here:
- `pdb_samples/*.pdb`
- `mdp/ions.mdp`
- `mdp/step4.0_minimization.mdp`

Local Amber topology bundle kept here for reference:
- `amberff/topol.top`
- `amberff/topol*.itp`
- `amberff/posre*.itp`

The cells below keep the old batch style visible:
- `GPU_INDEX`
- `NUM_GPUS`
- `ONLY_PDB`
- `TRUNCATE_FILE`
- per-file retries for step4 minimization


In [ ]:
# Gromacs.1 - Set local paths and validate the copied inputs.
# Details:
# - Uses only files stored in /media/volume/sirna-features/gromacs.
# - Fails early if gmx is not available on PATH or if required local inputs are missing.

from pathlib import Path
import os
import shutil
import subprocess

DATASET_DIR = Path("/media/volume/sirna-features/gromacs")
MDP_DIR = DATASET_DIR / "mdp"
AMBER_DIR = DATASET_DIR / "amberff"
PDB_DIRECTORIES = [
    DATASET_DIR / "pdb_samples",
]
RUNS_DIR = DATASET_DIR / "runs"
RUNS_DIR.mkdir(exist_ok=True)

LOCAL_PDBS = sorted(
    path
    for pdb_dir in PDB_DIRECTORIES
    if pdb_dir.exists()
    for path in pdb_dir.glob("*.pdb")
)
REQUIRED_PATHS = PDB_DIRECTORIES + [
    MDP_DIR,
    AMBER_DIR,
    MDP_DIR / "ions.mdp",
    MDP_DIR / "step4.0_minimization.mdp",
]

missing = [str(path) for path in REQUIRED_PATHS if not path.exists()]
if missing:
    raise FileNotFoundError("Missing local Gromacs inputs:\n" + "\n".join(missing))
if not LOCAL_PDBS:
    raise FileNotFoundError("No local PDB files were found under: " + ", ".join(str(path) for path in PDB_DIRECTORIES))

GMX_BIN = shutil.which("gmx")
if not GMX_BIN:
    raise FileNotFoundError("gmx is not on PATH. Load or install Gromacs before running this notebook.")

GPU_INDEX = int(os.environ.get("GPU_INDEX", os.environ.get("SLURM_ARRAY_TASK_ID", "0")))
NUM_GPUS = int(os.environ.get("NUM_GPUS", os.environ.get("SLURM_ARRAY_TASK_COUNT", "6")))
USE_GPU_ID_FLAG = False
GMX_GPU_ID = os.environ.get("GMX_GPU_ID", str(GPU_INDEX))
ONLY_PDB = None
TRUNCATE_FILE = None
SKIP_COMPLETED_STEP4 = True
SAMPLE_ONLY = False

print("Gromacs binary:", GMX_BIN)
print(subprocess.run([GMX_BIN, "--version"], capture_output=True, text=True).stdout.splitlines()[:3])
print("Batch settings:")
print(" - GPU_INDEX =", GPU_INDEX)
print(" - NUM_GPUS =", NUM_GPUS)
print(" - USE_GPU_ID_FLAG =", USE_GPU_ID_FLAG)
print(" - GMX_GPU_ID =", GMX_GPU_ID)
print(" - ONLY_PDB =", ONLY_PDB)
print(" - TRUNCATE_FILE =", TRUNCATE_FILE)
print(" - SKIP_COMPLETED_STEP4 =", SKIP_COMPLETED_STEP4)
print(" - SAMPLE_ONLY =", SAMPLE_ONLY)
print("Local PDB directories:")
for pdb_dir in PDB_DIRECTORIES:
    print(" -", pdb_dir)
print("Local PDB files:")
for path in LOCAL_PDBS:
    print(" -", path.name)


In [ ]:
# Gromacs.2 - Define the local structure setup and step4 minimization path.
# Details:
# - Starts directly from a PDB file.
# - Recreates the core run sequence used in the original MD notebooks:
#   pdb2gmx -> editconf -> solvate -> grompp(ions) -> genion -> make_ndx -> grompp(minimization) -> mdrun
# - Keeps the original step4 retry pattern:
#   run minimization, then retry mdrun up to 3 times if it does not converge cleanly.
# - Writes each run into its own folder under runs/<sample_name>/.

from pathlib import Path
import os
import re
import shutil
import subprocess
import time

FORCEFIELD_INPUT = "6\n1\n"
GENION_GROUP_INPUT = "14\n"
MAKE_NDX_INPUT = "name 19 SOLV\n1 | 12\nname 20 SOLU\nq\n"

os.environ["GMX_MAXBACKUP"] = "-1"
os.environ["GMX_MAXCONSTRWARN"] = "-1"

def run_command(command, cwd: Path, input_text: str | None = None) -> subprocess.CompletedProcess:
    result = subprocess.run(
        command,
        cwd=str(cwd),
        input=input_text,
        text=True,
        capture_output=True,
        check=False,
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed in {cwd}: {' '.join(command)}\n\nSTDOUT:\n{result.stdout}\n\nSTDERR:\n{result.stderr}"
        )
    return result

def run_mini(command, cwd: Path, input_text: str | None = None) -> bool:
    result = subprocess.run(
        command,
        cwd=str(cwd),
        input=input_text,
        text=True,
        capture_output=True,
        check=False,
    )
    output = result.stdout + result.stderr
    for line in output.splitlines():
        lowered = line.lower()
        if (
            "steepest descents converged to" in lowered
            or "steepest descents did not converge" in lowered
            or "largest distance between excluded atoms" in lowered
            or "fatal" in lowered
            or "error" in lowered
        ):
            print(line)
        match = re.search(r"(\d+) steps", line)
        if match and "steepest descents converged to" in lowered:
            steps = int(match.group(1))
            return steps == 5001
    return False

def fresh_run_dir(sample_name: str) -> Path:
    run_dir = RUNS_DIR / sample_name
    if run_dir.exists():
        shutil.rmtree(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_dir

def copy_local_mdp(run_dir: Path) -> None:
    for source in [
        MDP_DIR / "ions.mdp",
        MDP_DIR / "step4.0_minimization.mdp",
    ]:
        if source.exists():
            shutil.copy2(source, run_dir / source.name)

def run_structure_setup(input_pdb: Path, run_dir: Path) -> None:
    copy_local_mdp(run_dir)
    for pattern in ("*.gro", "*.tpr", "index.ndx", "topol.top", "topol_*.itp", "posre*.itp"):
        for existing in run_dir.glob(pattern):
            existing.unlink()

    run_command(
        [GMX_BIN, "pdb2gmx", "-f", input_pdb.name, "-o", "structure_processed.gro", "-p", "topol.top", "-i", "posre.itp"],
        cwd=run_dir,
        input_text=FORCEFIELD_INPUT,
    )
    run_command(
        [GMX_BIN, "editconf", "-f", "structure_processed.gro", "-o", "structure_box.gro", "-c", "-d", "1.0", "-bt", "cubic"],
        cwd=run_dir,
    )
    run_command(
        [GMX_BIN, "solvate", "-cp", "structure_box.gro", "-cs", "spc216.gro", "-o", "structure_solv.gro", "-p", "topol.top"],
        cwd=run_dir,
    )
    run_command(
        [GMX_BIN, "grompp", "-f", "ions.mdp", "-c", "structure_solv.gro", "-p", "topol.top", "-o", "ions.tpr", "-maxwarn", "3"],
        cwd=run_dir,
    )
    run_command(
        [GMX_BIN, "genion", "-s", "ions.tpr", "-o", "structure_solv_ions.gro", "-p", "topol.top", "-pname", "NA", "-nname", "CL", "-neutral", "-conc", "0.15", "-seed", "12345"],
        cwd=run_dir,
        input_text=GENION_GROUP_INPUT,
    )
    run_command(
        [GMX_BIN, "make_ndx", "-f", "structure_solv_ions.gro", "-o", "index.ndx"],
        cwd=run_dir,
        input_text=MAKE_NDX_INPUT,
    )

def run_step4_minimization(run_dir: Path) -> dict:
    grompp_command = [
        GMX_BIN, "grompp", "-v", "-f", "step4.0_minimization.mdp", "-o", "step4.0_minimization.tpr",
        "-c", "structure_solv_ions.gro", "-r", "structure_solv_ions.gro",
        "-p", "topol.top", "-n", "index.ndx", "-maxwarn", "5",
    ]
    run_mini(grompp_command, cwd=run_dir)

    mdrun_command = [GMX_BIN, "mdrun", "-v", "-deffnm", "step4.0_minimization", "-ntmpi", "1"]
    if USE_GPU_ID_FLAG:
        mdrun_command.extend(["-gpu_id", str(GMX_GPU_ID)])
    # step4.0_minimization.mdp keeps nsteps = 5000 as the final target here.
    # Gromacs often branches during minimization for efficiency, so a run may stop early
    # around 3000 steps, or even closer to 1000, instead of walking the full 5000-step path.
    # The highest reproducibility here came from keeping the full 5000 steps for all samples
    # without taking that shortcut, so run_mini() catches those short-run keyword patterns
    # and retries mdrun up to 3 times to try to land on the full 5000-step minimization.
    tries = 0
    converged = run_mini(mdrun_command, cwd=run_dir)
    while tries < 3 and not converged:
        tries += 1
        converged = run_mini(mdrun_command, cwd=run_dir)

    return {
        "converged": bool(converged),
        "tries_used": tries + 1,
        "final_gro": str(run_dir / "step4.0_minimization.gro"),
        "final_tpr": str(run_dir / "step4.0_minimization.tpr"),
    }

def get_pdb_files(
    pdb_directory: Path,
    gpu_index: int,
    num_gpus: int,
    only_pdb: str | None = None,
    truncate_file: str | None = None,
    skip_completed_step4: bool = True,
    sample_only: bool = False,
) -> list[Path]:
    if sample_only:
        pdb_files = list(LOCAL_PDBS)
    else:
        sample_names = {sample.name for sample in LOCAL_PDBS}
        pdb_files = sorted(
            path for path in pdb_directory.glob("*.pdb")
            if path.name not in sample_names
        )

    if only_pdb:
        pdb_files = [path for path in pdb_files if path.name == only_pdb]

    if truncate_file:
        truncate_index = next((i for i, path in enumerate(pdb_files) if path.name == truncate_file), len(pdb_files))
        pdb_files = pdb_files[:truncate_index]

    if skip_completed_step4:
        pending = []
        for path in pdb_files:
            final_gro = RUNS_DIR / path.stem / "step4.0_minimization.gro"
            if not final_gro.exists():
                pending.append(path)
        pdb_files = pending

    if not pdb_files:
        return []

    total_rows = len(pdb_files)
    portion_size = total_rows // num_gpus
    start_idx = gpu_index * portion_size
    end_idx = (gpu_index + 1) * portion_size if gpu_index < (num_gpus - 1) else total_rows
    return pdb_files[start_idx:end_idx]

def run_gromacs_pipeline(input_pdb: Path) -> dict:
    sample_name = input_pdb.stem
    run_dir = fresh_run_dir(sample_name)
    shutil.copy2(input_pdb, run_dir / input_pdb.name)
    start_time = time.time()
    run_structure_setup(input_pdb, run_dir)
    step4_result = run_step4_minimization(run_dir)

    return {
        "sample_name": sample_name,
        "run_dir": str(run_dir),
        "input_pdb": str(input_pdb),
        "elapsed_seconds": round(time.time() - start_time, 2),
        **step4_result,
    }
